In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import copy
import time
from astropy.time import Time
import numpy as np
import os

from orbitalsim import diagnostic_calculators
from orbitalsim.universal_constants import UniversalConstants
from orbitalsim.celestial_body import CelestialBody
from orbitalsim.planets import Planets
from orbitalsim.diagnostic_calculators import *
from orbitalsim.simulation import NBodySimulation
from orbitalsim.set_positions import PositionsSetter

from orbitalsim.integrators.symplectic_euler import SymplecticEulerIntegrator
from orbitalsim.integrators.yoshida_4th_order import Yoshida4thOrderIntegrator
from orbitalsim.integrators.forward_euler import ForwardEuler
from orbitalsim.integrators.runge_kutta_4 import RungeKutta4Integrator
from orbitalsim.integrators.velocity_verlet import VelocityVerletIntegrator

from notebooks.results import *

In [ ]:
os.makedirs("results",exist_ok=True)

In [ ]:
import urllib.request

url = "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00010.tpc"
urllib.request.urlretrieve(url, "../pck00010.tpc")
print("Downloaded successfully!")

url = "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de431.tpc"
urllib.request.urlretrieve(url, "../gm_de431.tpc")
print("Downloaded successfully!")

In [ ]:
class SimulationRunner:

    integrator = RungeKutta4Integrator()

    sun = CelestialBody(1.32712440018e20, 1.989e30, np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 0.0]), name="Sun")

    all_bodies = copy.deepcopy(Planets().allPlanets)
    all_bodies.append(sun)

    center_of_mass = sum(body.mass * body.position for body in all_bodies) / sum(body.mass for body in all_bodies)

    for body in all_bodies:
        body.position = body.position - center_of_mass

    total_momentum = sum(body.mass * body.velocity for body in all_bodies)

    for body in all_bodies:
        if body.name == "Sun":
            body.velocity = body.velocity - total_momentum / body.mass

    simulation = NBodySimulation(integrator, bodies=all_bodies)

    traj, saved_velocities, *_ = simulation.simulator(20000, 500)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    for i, body in enumerate(simulation.bodies):
        ax.plot(
            traj[:, i, 0],
            traj[:, i, 1],
            traj[:, i, 2],
            label = body.name
        )

        # final pos
        ax.scatter(
            traj[-1, i, 0],
            traj[-1, i, 1],
            traj[-1, i, 2],
        )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.legend()

    plt.show()


In [2]:
class SimulationDebugRunner:

    sun = CelestialBody(1.32712440018e20, 1.32712440018e20 / UniversalConstants.G, np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 0.0]), name="Sun", horizons_id='10')

    all_bodies = copy.deepcopy(Planets().allPlanets)
    all_bodies.append(sun)

    center_of_mass = sum(body.mass * body.position for body in all_bodies) / sum(body.mass for body in all_bodies)

    for body in all_bodies:
        body.position = body.position - center_of_mass

    total_momentum = sum(body.mass * body.velocity for body in all_bodies)

    for body in all_bodies:
        if body.name == "Sun":
            body.velocity = body.velocity - total_momentum / body.mass

    start_time = "2000-01-01 00:00:00"
    epoch = Time(start_time).tdb.jd
    PositionsSetter.set_positions_at_time(all_bodies,epoch)

    # print(diagnostic_calculators.two_simulation_position_comparison(all_bodies, 50000, Yoshida4thOrderIntegrator(), RungeKutta4Integrator(), 10000, 100))

    print(diagnostic_calculators.horizon_data_position_comparison(all_bodies, start_time, 2000, Yoshida4thOrderIntegrator(), 10))

[3.69389158e-01 9.34745662e-02 1.65378989e-01 1.25530143e-02
 1.71317285e-03 7.42524723e-04 2.44145393e-03 4.20074387e-03
 1.27601887e-04]


In [ ]:
class TwoBodyEarthSunComparison:

    integrator = ForwardEuler()
    dt = 50000

    sun = CelestialBody(1.32712440018e20, 1.32712440018e20 / UniversalConstants.G, np.array([-4.4930949e5, 0.0, 0.0]), np.array([0.0, -0.0895, 0.0]), name="Sun", horizons_id='10')
    earth = copy.deepcopy(Planets().earth)
    r = np.linalg.norm(earth.position - sun.position)
    earth.velocity = sun.velocity + np.array([0.0, Planets.true_circular_velocities(r, earth.mass, sun.mass), 0.0])
    all_bodies = [earth, copy.deepcopy(sun)]

    simulation = NBodySimulation(integrator, bodies=all_bodies)

    start_time = time.perf_counter()
    traj, saved_velocities, *_ = simulation.simulator(50000000, dt)
    end_time = time.perf_counter()
    total_time = end_time - start_time
    print(f"Forward Euler: {total_time}")

    r = np.linalg.norm(traj[0, 0] - traj[0, 1])

    print(r)

    orbital_period = 2 * np.pi * np.sqrt(r**3 / (UniversalConstants.G * (all_bodies[0].mass + all_bodies[1].mass)))

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    for i, body in enumerate(simulation.bodies):
        ax.plot(
            traj[:, i, 0],
            traj[:, i, 1],
            traj[:, i, 2],
            label = body.name
        )

        # final pos
        ax.scatter(
            traj[-1, i, 0],
            traj[-1, i, 1],
            traj[-1, i, 2],
        )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.legend()

    plt.show()

    df = diagnostic_calculators.analyze_two_body(traj, saved_velocities, simulation.bodies, dt, orbital_period)
    #df.to_csv("results/raw_data/forward_euler_diagnostics_v3.csv", index=False)

    integrator = SymplecticEulerIntegrator()

    earth = copy.deepcopy(Planets().earth)
    r = np.linalg.norm(earth.position - sun.position)
    earth.velocity = sun.velocity + np.array([0.0, Planets.true_circular_velocities(r, earth.mass, sun.mass), 0.0])
    all_bodies = [earth, copy.deepcopy(sun)]

    simulation = NBodySimulation(integrator, bodies=all_bodies)

    start_time = time.perf_counter()
    traj, saved_velocities, *_ = simulation.simulator(50000000, dt)
    end_time = time.perf_counter()
    total_time = end_time - start_time
    print(f"Symplectic Euler: {total_time}")

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    for i, body in enumerate(simulation.bodies):
        ax.plot(
            traj[:, i, 0],
            traj[:, i, 1],
            traj[:, i, 2],
            label = body.name
        )

        # final pos
        ax.scatter(
            traj[-1, i, 0],
            traj[-1, i, 1],
            traj[-1, i, 2],
        )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.legend()

    plt.show()

    df = diagnostic_calculators.analyze_two_body(traj, saved_velocities, simulation.bodies, dt, orbital_period)
    #df.to_csv("results/raw_data/symplectic_euler_diagnostics_v3.csv", index=False)

    integrator = RungeKutta4Integrator()

    earth = copy.deepcopy(Planets().earth)
    r = np.linalg.norm(earth.position - sun.position)
    earth.velocity = sun.velocity + np.array([0.0, Planets.true_circular_velocities(r, earth.mass, sun.mass), 0.0])
    all_bodies = [earth, copy.deepcopy(sun)]

    simulation = NBodySimulation(integrator, bodies=all_bodies)

    start_time = time.perf_counter()
    traj, saved_velocities, *_ = simulation.simulator(50000000, dt)
    end_time = time.perf_counter()
    total_time = end_time - start_time
    print(f"RK4: {total_time}")

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    for i, body in enumerate(simulation.bodies):
        ax.plot(
            traj[:, i, 0],
            traj[:, i, 1],
            traj[:, i, 2],
            label = body.name
        )

        # final pos
        ax.scatter(
            traj[-1, i, 0],
            traj[-1, i, 1],
            traj[-1, i, 2],
        )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.legend()

    plt.show()

    df = diagnostic_calculators.analyze_two_body(traj, saved_velocities, simulation.bodies, dt, orbital_period)
    #df.to_csv("results/raw_data/runge_kutta_4_diagnostics_v3.csv", index=False)

    integrator = VelocityVerletIntegrator()

    earth = copy.deepcopy(Planets().earth)
    r = np.linalg.norm(earth.position - sun.position)
    earth.velocity = sun.velocity + np.array([0.0, Planets.true_circular_velocities(r, earth.mass, sun.mass), 0.0])
    all_bodies = [earth, copy.deepcopy(sun)]

    simulation = NBodySimulation(integrator, bodies=all_bodies)

    start_time = time.perf_counter()
    traj, saved_velocities, *_ = simulation.simulator(50000000, dt)
    end_time = time.perf_counter()
    total_time = end_time - start_time
    print(f"Velocity Verlet: {total_time}")

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    for i, body in enumerate(simulation.bodies):
        ax.plot(
            traj[:, i, 0],
            traj[:, i, 1],
            traj[:, i, 2],
            label = body.name
        )

        # final pos
        ax.scatter(
            traj[-1, i, 0],
            traj[-1, i, 1],
            traj[-1, i, 2],
        )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.legend()

    plt.show()

    df = diagnostic_calculators.analyze_two_body(traj, saved_velocities, simulation.bodies, dt, orbital_period)
    #df.to_csv("results/raw_data/velocity_verlet_diagnostics_v3.csv", index=False)

    integrator = Yoshida4thOrderIntegrator()

    earth = copy.deepcopy(Planets().earth)
    r = np.linalg.norm(earth.position - sun.position)
    earth.velocity = sun.velocity + np.array([0.0, Planets.true_circular_velocities(r, earth.mass, sun.mass), 0.0])
    all_bodies = [earth, copy.deepcopy(sun)]

    simulation = NBodySimulation(integrator, bodies=all_bodies)

    start_time = time.perf_counter()
    traj, saved_velocities, *_ = simulation.simulator(50000000, dt)
    end_time = time.perf_counter()
    total_time = end_time - start_time
    print(f"Yoshida: {total_time}")

    r = np.linalg.norm(all_bodies[0].position - all_bodies[1].position)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    for i, body in enumerate(simulation.bodies):
        ax.plot(
            traj[:, i, 0],
            traj[:, i, 1],
            traj[:, i, 2],
            label = body.name
        )

        # final pos
        ax.scatter(
            traj[-1, i, 0],
            traj[-1, i, 1],
            traj[-1, i, 2],
        )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.legend()

    plt.show()

    df = diagnostic_calculators.analyze_two_body(traj, saved_velocities, simulation.bodies, dt, orbital_period)
    #df.to_csv("results/raw_data/yoshida_4th_order_diagnostics_v3.csv", index=False)


In [ ]:
class TwoBodyDataAnalysis:

    def __init__(self):
        self.methods = {
            #'Forward Euler': pd.read_csv('results/raw_data/forward_euler_diagnostics_v3.csv'),
            #'Symplectic Euler': pd.read_csv('results/raw_data/symplectic_euler_diagnostics_v3.csv'),
            #'Runge-Kutta 4': pd.read_csv('results/raw_data/runge_kutta_4_diagnostics_v3.csv'),
            #'Velocity Verlet': pd.read_csv('results/raw_data/velocity_verlet_diagnostics_v3.csv'),
            'Yoshida 4th-Order': pd.read_csv('results/raw_data/yoshida_4th_order_diagnostics_v3.csv')
        }

    def plot_radius(self, use_log_scale=False, save=False):
        plt.figure(figsize=(10, 6))

        for name, df in self.methods.items():
            plt.plot(df['time_years'], df['radius_meters'],label=name,alpha=0.8)

        plt.title('Orbital Radius vs. Time - dt = 50,000', fontsize=14)
        plt.xlabel('Time (Years)', fontsize=12)
        plt.ylabel('Orbital Radius (Meters)', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.6)

        if use_log_scale:
            plt.yscale('log')
            plt.ylabel('Orbital Radius (Meters) [Log Scale]', fontsize=12)

        plt.legend(loc='best')
        plt.tight_layout()

        if save:
            plt.savefig('radius_vs_time.png', dpi=300)

        plt.show()

    def plot_velocity_error(self, use_log_scale=False, save=False):
        plt.figure(figsize=(10, 6))

        for name, df in self.methods.items():
            plt.plot(df['time_years'], df['speed_error_m_per_s'],label=name,alpha=0.8)

        plt.title('Velocity Error vs. Time - dt = 50,000', fontsize=14)
        plt.xlabel('Time (Years)', fontsize=12)
        plt.ylabel('Velocity Error (Meters/Second)', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.6)

        if use_log_scale:
            plt.yscale('log')
            plt.ylabel('Velocity Error (Meters/Second) [Log Scale]', fontsize=12)

        plt.legend(loc='best')
        plt.tight_layout()

        if save:
            plt.savefig('velocity_error_vs_time.png', dpi=300)

        plt.show()

analysis = TwoBodyDataAnalysis()
analysis.plot_radius()
analysis.plot_velocity_error()

Currently, the architecture of the project works by iterating through and body at each step, and then iterating through each body again to calculate the forces acting on all of them and in turn calculate the new positions and velocities via one of the integrators (Forward Euler, Symplectic Euler, Runge-Kutta 4, Velocity Verlet, Yoshida 4th-Order). It then repeats this for each step until it is complete, at which point it returns an array of each position at each step, the final positions, and the maximum and minimums of energy and linear and angular momentum.

This data can then be either visualized with matplotlib to show a trajectory or compared via other results. To compare with other results there is two_simulation_position_comparison that will compare one integrator with another where each may have a different time step size with the same ending time, or there is horizon_data_position_comparison, that takes an initial start time in a readable format for JPL Horizons, computes the end time based on the steps and step size, then compares the ending position of the simulation to what JPL Horizons has recorded for that time.

This simulation currently only has the sun and the eight planets and uses barycentric coordinates, treating the sun as a dynamical body. Masses for each planet are currently imprecise, increasing error when compared with Horizons data. Initial positions and velocities are assigned via the set_positions_at_time method.

For a brief verification, we will conduct a two-body test using Earth and the Sun, giving Earth a circular orbit such that $v=\sqrt{\frac{GM}{r}}$ and $T=2\pi\sqrt{\frac{r^3}{GM}}$. We will compare the known analytical solutions to the simulation results to compare integrator accuracy. Analyzing over ~73,000 years with a time-step size of 50,000 seconds, we see the following behavior:

Forward Euler: Explodes in both velocity and positional error. Notably, it appears to exhibit an exponential decay in the growth of its error, particularly for velocity error, with approximately 86% of the total error (21043 m/s of 24486 m/s) being accumulated in the first 5% of run-time (4000 years of 79216 years). Positional error does not as explicitly demonstrate such asymptotic behavior over this time span, though the graph implies that such behavior may emerge given more time. It is also of note that both errors oscilate rapidly, with the bounds of oscillation growing as time goes on, though just as with the total error, there appears to be similar asymptotic behavior for the bounds of oscillation, and, once again, this is most especially apparent in the graph of velocity error. It had a total runtime of 3751 seconds, which decreased to 1872 seconds upon doubling the time-step, and the double in timestep approximately doubled the error.
<div style="height: 30px;"></div>
<img src="results/visualizations/forward_euler_circular_orbit_radius_vs_time.png" width="450" hspace="40"> <img src="results/visualizations/forward_euler_circular_orbit_velocity_error_vs_time.png" width="450">

Symplectic Euler: Oscillates with consistent bounds for both velocity and positional error. Velocity error remains strictly bound by approximately 149 m/s and orbital radius error is strictly bound by approximately $7.5 \cdot 10^8$ meters. The waves of their errors remain compliments of each other, with the crests of the velocity error corresponding to the troughs of the orbital radius error and vice versa. The period of both is approximately 11090 years. The error does not strictly oscilate, with minor fluctuations, but the broad trend is overwhelmingly a consistent wave function. It had a total runtime of 3789 seconds, which decreased to 2002 upon doubling the time-step, and the double in time-step approximately doubled the error.

<div style="height: 30px;"></div>
<img src="results/visualizations/symplectic_euler_circular_orbit_radius_vs_time.png" width="450" hspace="40"> <img src="results/visualizations/symplectic_euler_circular_orbit_velocity_error_vs_time.png" width="450">

Runge-Kutta 4: Both velocity error and orbital radius error increase as simple linear functions. Specifically, the approximate error of orbital radius may be represented at year $t$ by $2.55t$ meters, and the approximate velocity error by $2.52 \cdot 10^{-7} t$. The orbital radius is constantly decreasing. It had a total runtime of 6268 seconds, which decreased to 3325 upon doubling the timestep, and the double in time-step multiplied the error by approximately 16.

<div style="height: 30px;"></div>
<img src="results/visualizations/runge_kutta_4_circular_orbit_radius_vs_time.png" width="450" hspace="40"> <img src="results/visualizations/runge_kutta_4_circular_orbit_velocity_error_vs_time.png" width="450">

Velocity Verlet: Behaves similar to Symplectic Euler, being symplectic as well, but with a significantly smaller error and a longer period that does not finish by the end of the 80,000 years. It also appears that its error functions start at the extrema of the waves, with the velocity error starting at the crest and the orbital radius error starting at the trough. The velocity error is bounded by approximately 1.5 m/s and the orbital radius error is bounded by approximately $7.5 \cdot 10^{6}$ meters, both reaching their maximum errors at approximately 60,000 years. It had a total run time of 4444 seconds, which decreased to 2425 upon doubling the timestep, and the double in time-step approximately quadrupled the error.

<div style="height: 30px;"></div>
<img src="results/visualizations/velocity_verlet_circular_orbit_radius_vs_time.png" width="450" hspace="40"> <img src="results/visualizations/velocity_verlet_circular_orbit_velocity_error_vs_time.png" width="450">

Yoshida 4th Order: Being built off of Velocity Verlet, it is unsurprising that there appears to be the same structure of a wave, with what may be a crest at approximately 60,000 years as well. There seems to be a large increase in the rate of change of error at roughly 37,000 years for both velocity and orbital radius. The two errors compliment each other as well. It should be noted that the plot of the errors is significantly less straightforward than the for the other integrators. Yoshida introduces variance with its timestep coefficients designed to reduce error, which produces significant noise, but it ultimately produces a wave similar to Verlet when looking at the broader graph, and succeeds greatly in reducing error. It stays within less than a meter of the intended orbital radius and nanometers per second of velocity error. It had a total runtime of 7680 seconds, which decreased to 3663 upon doubling the timestep, and the double in time-step multiplied the error by approximately 16.

<div style="height: 30px;"></div>
<img src="results/visualizations/yoshida_4th_order_circular_orbit_radius_vs_time.png" width="450" hspace="40"> <img src="results/visualizations/yoshida_4th_order_circular_orbit_velocity_error_vs_time.png" width="450">

Overall, Yoshida is by far the most accurate, with the relative error of the next most accurate, Runge-Kutta 4, being about 5 to 6 orders of magnitude less accurate. It is also symplectic, meaning that even as the duration increases its error will remain bounded by that very small order while another integrator like RK4 or Forward Euler could accumulate error leading to massive energy drift. The same holds true for Velocity Verlet and Symplectic Euler, both being symplectic. The symplecticity of these integrators preserves the geometric phase-space volume of the Hamiltonian systems. In other words, these symplectic integrators ensure that the "map" or phase space tracking both the position and momentum of each particle in the system conserves its total "area," while standard solvers like Runge-Kutta 4 slowly distort it. These symplectic integrators do not actually conserve the exact Hamiltonian, but rather what is known as a Shadow Hamiltonian, a slightly perturbed energy function. However, it is worth noting that while Yoshida is by far the most accurate, it does have the greatest runtime, though the massive increase in accuracy would be worth it in many situations.